# Run a 20-page PDF through three chunking strategies, then prove which one wins on retrieval recall for a five-question evaluation set

The PM has been asking 'why does the bot not know what is on page 12' for a week. You looked at the bot logs and the retriever is returning the wrong chunks for every question whose answer crosses a page break. Default 1000-character fixed-size splitting is a sensible default and the wrong default for this document. You have one session to try three strategies, score them on the same five test questions, and ship a one-paragraph 'here is what we are using going forward' note the PM can forward to the team.

The hands-on work:

- Upload a 20-page product PDF into a Unity Catalog Volume at `workspace.default.labrun_chunking.source_docs` and parse it page by page with `pypdf`.
- Implement three chunking strategies as functions: fixed-size 512 with 50 overlap, `langchain.text_splitters.RecursiveCharacterTextSplitter` with paragraph + line + sentence separators, and semantic chunking driven by sentence-embedding cosine similarity.
- Embed every chunk with `databricks-gte-large-en` (1024-dim) via the Foundation Model API and persist three Delta tables.
- Compute recall@3 for each strategy on the same five-question evaluation set, write the per-strategy scores to a results table, and pick the winner.
- Write a one-paragraph justification that names the structural property of this PDF (page breaks, paragraph boundaries, semantic shifts) that explains the win.

Confirming the winning strategy comes from a measurable recall@3 lift, not a vibe, is what proves your retrieval layer choice is defensible.

**Time.** Plan for about 60 minutes hands-on. The semantic-chunking loop embeds every sentence individually and that can push you past 70 minutes the first time you tune the threshold. Budget up to 100 minutes for the session window.

**Cost.** Roughly $0.00 to $0.03 per session, all Foundation Model API embedding calls. About 165 embedding calls total: 150 chunk embeddings across the three strategies, 15 question embeddings (5 questions x 3 strategies). At a few hundred tokens per call against `databricks-gte-large-en`, the token spend is fractions of a cent on Free Edition's pay-per-token pricing. Your morning coffee cost 100x more.

**Fixture honesty.** The lab ships ONE specific PDF as the input. The five evaluation questions and their ground-truth keywords are pre-computed against that exact PDF, so the recall numbers are deterministic. If you swap the PDF, the scores change and the brief explicitly says so. The setup cell prints the fixture URL and the path you should use; you can grab the PDF via `dbutils.fs.cp` from inside the notebook or upload it manually through the workspace UI.

This lab maps to Databricks GenAI Engineer Associate Domain 2: Data Preparation (~14% of exam weight, provisional).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks and the runtime dependencies this lab needs. Pinned
# versions per LAB-CREATION-BLUEPRINT.md build rules. Never use unpinned
# installs. langchain-text-splitters is the slim splitter-only package; we
# do NOT need full langchain or langchain-community here.

!pip install --quiet labrun-checks==0.7.0 databricks-sdk==0.40.0 openai==1.54.0 pypdf==4.3.1 langchain-text-splitters==0.3.0 numpy==1.26.4

In [ ]:
# Imports and per-lab constants. UC identifiers (schema, volume, tables) use
# snake_case under the labrun_ prefix because Unity Catalog identifiers prefer
# snake_case (hyphens force backtick-quoting everywhere downstream).

import atexit
import getpass
import io
import json
import re
import sys
import time

import numpy as np
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "document-loading-and-chunking-strategies"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[2].slug exactly

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_chunking"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"
LAB_VOLUME = "source_docs"
LAB_VOLUME_FQN = f"{LAB_SCHEMA_FQN}.{LAB_VOLUME}"
LAB_VOLUME_PATH = f"/Volumes/{CATALOG}/{PARENT_SCHEMA}/{LAB_SCHEMA}/{LAB_VOLUME}"

RAW_TEXT_TABLE = f"{LAB_SCHEMA_FQN}.raw_text"
CHUNKS_FIXED_TABLE = f"{LAB_SCHEMA_FQN}.chunks_fixed"
CHUNKS_RECURSIVE_TABLE = f"{LAB_SCHEMA_FQN}.chunks_recursive"
CHUNKS_SEMANTIC_TABLE = f"{LAB_SCHEMA_FQN}.chunks_semantic"
EVAL_SET_TABLE = f"{LAB_SCHEMA_FQN}.eval_set"
RESULTS_TABLE = f"{LAB_SCHEMA_FQN}.chunking_results"

EMBEDDING_MODEL = "databricks-gte-large-en"
EMBEDDING_DIM = 1024
EMBED_BATCH_SIZE = 32

# Fixture: the lab uses a single specific Databricks-hosted public PDF so the
# evaluation set ground-truth keywords are deterministic. The setup cell
# prints the canonical fixture URL and the volume path the student should
# place the file at. The eval set below was hand-built against this fixture.
FIXTURE_PDF_FILENAME = "product_overview.pdf"
FIXTURE_PDF_PATH = f"{LAB_VOLUME_PATH}/{FIXTURE_PDF_FILENAME}"
FIXTURE_PDF_URL = "https://labrun-public-fixtures.s3.amazonaws.com/databricks-genai/lab-03/product_overview.pdf"

# Five evaluation questions with their ground-truth keyword (must appear in
# at least one top-3 chunk for the chunk to count as a hit). Keywords are
# stable substrings that appear in the source PDF and are unique enough that
# substring matching gives an honest signal without needing exact span IDs.
EVAL_SET = [
    (1, "What is the maximum file size that the platform supports for ingestion?", "5 gigabytes"),
    (2, "Which authentication mechanism is recommended for service-to-service calls?", "OAuth 2.0 client credentials"),
    (3, "How does the platform handle schema evolution on Delta tables?", "automatic schema evolution"),
    (4, "What is the default retention window for deleted records before vacuum removes them?", "seven days"),
    (5, "Which region does the platform recommend for the primary metastore?", "us-east-1"),
]

In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials via the SDK, resolve
# the Starter SQL warehouse, and construct the OpenAI-compatible client we
# point at the Foundation Model API for embedding calls. This cell must
# succeed before the manifest cell runs anything, per LAB-CREATION-BLUEPRINT
# section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL (https://...azuredatabricks.net or .cloud.databricks.com): ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired. CurrentUser.me() failed:")
    print(f"  {exc}")
    print()
    print("Refresh your workspace URL and PAT, then re-run this cell.")
    raise SystemExit(1)

CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found in this workspace. Free Edition ships with")
    print("a Starter warehouse by default; if it has been deleted, recreate it")
    print("from the SQL > Warehouses page before re-running this cell.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

# OpenAI-compatible client pointed at the Foundation Model API serving
# endpoints base URL. The PAT is the bearer token. The chunking work only
# calls embeddings.create; no LLM completions in this lab.
openai_client = OpenAI(
    api_key=databricks_token,
    base_url=f"{databricks_host}/serving-endpoints",
)

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    """Submit a SQL statement to the warehouse and return the parsed result.

    Polls statement state up to wait_seconds, raises on FAILED or CANCELED,
    returns a dict {columns: [...], rows: [[...]]} on SUCCEEDED.
    """
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid,
        statement=statement,
        wait_timeout="50s",
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(
                f"SQL did not finish in {wait_seconds}s. Last state: {state}. "
                f"The Starter warehouse may still be waking up; re-run the cell."
            )
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)

    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


def embed_texts(texts, batch_size=EMBED_BATCH_SIZE, sleep_between_batches=0.25):
    """Embed a list of strings with databricks-gte-large-en in batches.

    Returns a numpy array of shape (len(texts), 1024). Sleeps between batches
    to stay under the pay-per-token rate limit.
    """
    if not texts:
        return np.zeros((0, EMBEDDING_DIM), dtype=np.float32)
    out = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        resp = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
        for item in resp.data:
            out.append(item.embedding)
        if start + batch_size < len(texts):
            time.sleep(sleep_between_batches)
    arr = np.asarray(out, dtype=np.float32)
    return arr


def cosine_top_k(query_vec, chunk_matrix, k=3):
    """Return the indices of the top-k chunks by cosine similarity to query_vec."""
    q = query_vec.astype(np.float32)
    q_norm = np.linalg.norm(q)
    if q_norm == 0:
        return list(range(min(k, chunk_matrix.shape[0])))
    chunk_norms = np.linalg.norm(chunk_matrix, axis=1)
    chunk_norms = np.where(chunk_norms == 0, 1.0, chunk_norms)
    sims = (chunk_matrix @ q) / (chunk_norms * q_norm)
    top = np.argsort(-sims)[:k]
    return [int(i) for i in top]


register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")
print(f"Fixture PDF URL: {FIXTURE_PDF_URL}")
print(f"Target path inside the volume: {FIXTURE_PDF_PATH}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# Manifest is module-level and in reverse-creation order. No critical
# (hourly-billed) resources in this lab; UC storage and the Foundation
# Model API embedding calls are pay-per-use with the only ongoing cost
# being a few cents of tokens already spent. Per RESOURCE-SAFETY-SPEC
# Rule 4 the orphan scan blocks execution if any tagged resources from
# a prior session exist.


CLEANUP_MANIFEST = [
    CleanupResource(
        type="uc_managed_table",
        id=RESULTS_TABLE,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {RESULTS_TABLE}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=EVAL_SET_TABLE,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {EVAL_SET_TABLE}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=CHUNKS_SEMANTIC_TABLE,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {CHUNKS_SEMANTIC_TABLE}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=CHUNKS_RECURSIVE_TABLE,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {CHUNKS_RECURSIVE_TABLE}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=CHUNKS_FIXED_TABLE,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {CHUNKS_FIXED_TABLE}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=RAW_TEXT_TABLE,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {RAW_TEXT_TABLE}\"",
    ),
    CleanupResource(
        type="uc_volume_contents",
        id=LAB_VOLUME_FQN,
        region="databricks",
        cli_delete_command=f"databricks fs rm -r dbfs:{LAB_VOLUME_PATH}/",
    ),
    CleanupResource(
        type="uc_volume",
        id=LAB_VOLUME_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP VOLUME IF EXISTS {LAB_VOLUME_FQN}\"",
    ),
    CleanupResource(
        type="uc_schema",
        id=LAB_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {LAB_SCHEMA_FQN} CASCADE\"",
    ),
]


class DatabricksCleanupAdapter:
    """Inline adapter implementing the labrun-checks CleanupAdapter protocol
    against Databricks Unity Catalog via the SQL Statement Execution API.
    Targets uc_managed_table, uc_volume, uc_volume_contents, and uc_schema
    resources for the GenAI Engineer Associate cert track."""

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_volume":
            run_sql(f"DROP VOLUME IF EXISTS {rid}")
        elif rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
            except Exception:
                return
            for entry in listing:
                try:
                    if entry.is_directory:
                        w.files.delete_directory(entry.path)
                    else:
                        w.files.delete(entry.path)
                except Exception:
                    pass
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        else:
            raise ValueError(f"DatabricksCleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "uc_managed_table":
            catalog, schema, table = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{table}'"
            )
            result = run_sql(sql)
            return len(result["rows"]) > 0
        if rtype == "uc_volume":
            catalog, schema, volume = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.volumes "
                f"WHERE volume_catalog = '{catalog}' AND volume_schema = '{schema}' "
                f"AND volume_name = '{volume}'"
            )
            result = run_sql(sql)
            return len(result["rows"]) > 0
        if rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
                return len(listing) > 0
            except Exception:
                return False
        if rtype == "uc_schema":
            catalog, schema = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
            )
            result = run_sql(sql)
            return len(result["rows"]) > 0
        return False

    def tag_scan(self, credentials, lab_slug, region):
        """Return list of Unity Catalog identifiers tagged with the lab slug.

        Queries the per-entity tag views in system.information_schema and
        returns identifiers in dotted-FQN form so they can be matched against
        manifest ids."""
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
            (
                "SELECT catalog_name, schema_name, volume_name FROM system.information_schema.volume_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    return CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing UC objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources.")
    print("Run the cleanup cell at the bottom of this notebook first, or")
    print("drop them manually with the SQL commands shown by the cleanup")
    print("cell, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Stand up the schema and volume, get the PDF into the volume, parse it into a staging table

Three moves, each idempotent so a kernel restart mid-lab does not blow up.

1. `CREATE SCHEMA IF NOT EXISTS workspace.default.labrun_chunking`, then `ALTER SCHEMA ... SET TAGS ('labrun_lab_slug' = 'document-loading-and-chunking-strategies')`. The tag is what the orphan scan and cleanup audit read.
2. `CREATE VOLUME IF NOT EXISTS workspace.default.labrun_chunking.source_docs`. The volume is the UC-governed place the source PDF lives.
3. Drop the fixture PDF into the volume. Two paths work:
   - **From inside the notebook.** In a Databricks workspace notebook, `dbutils.fs.cp(FIXTURE_PDF_URL, FIXTURE_PDF_PATH)` pulls the file over HTTPS. The cell below also handles the case where `dbutils` is not in scope (for example, when running this notebook outside a Databricks workspace) by falling back to `urllib.request` and the Files API.
   - **From the workspace UI.** Open Catalog > workspace > default > labrun_chunking > source_docs > Upload to this volume, drag in your local copy of `product_overview.pdf`, click Upload.
4. Open the PDF with `pypdf.PdfReader`, walk each page, persist the page index and the extracted text to a small staging table at `workspace.default.labrun_chunking.raw_text` with schema `(page_index INT, page_text STRING)`.

Checkpoint 1 reads `raw_text` row count and the total character count across pages. A non-zero row count means the parser ran. A character count above 1000 means the parser actually pulled text (an empty-content PDF or a corrupted upload would clear the row count check but fail the content check).

In [ ]:
# Task 1: Create the schema, the volume, the raw_text table; pull the fixture
# PDF into the volume; parse it page by page and write rows.

# YOUR CODE: Run CREATE SCHEMA IF NOT EXISTS for LAB_SCHEMA_FQN, then
# ALTER SCHEMA ... SET TAGS ('labrun_lab_slug' = LAB_TAG_VALUE) via run_sql().

# YOUR CODE: Run CREATE VOLUME IF NOT EXISTS for LAB_VOLUME_FQN via run_sql().

# Get the fixture PDF into the volume. Try the dbutils.fs.cp path first; fall
# back to urllib + Files API if dbutils is not bound (running outside a
# Databricks workspace notebook).
try:
    dbutils.fs.cp(FIXTURE_PDF_URL, FIXTURE_PDF_PATH)  # type: ignore[name-defined]
    print(f"Copied via dbutils.fs.cp to {FIXTURE_PDF_PATH}")
except NameError:
    import urllib.request
    with urllib.request.urlopen(FIXTURE_PDF_URL) as resp:
        pdf_bytes = resp.read()
    w.files.upload(FIXTURE_PDF_PATH, io.BytesIO(pdf_bytes), overwrite=True)
    print(f"Uploaded via Files API to {FIXTURE_PDF_PATH} ({len(pdf_bytes)} bytes)")

# YOUR CODE: Run CREATE TABLE IF NOT EXISTS for RAW_TEXT_TABLE with schema
# (page_index INT, page_text STRING) USING DELTA via run_sql().

# YOUR CODE: Open the PDF at FIXTURE_PDF_PATH with PdfReader, iterate pages,
# extract text per page (page.extract_text() or empty string if None),
# escape single quotes for the INSERT, then INSERT all rows into RAW_TEXT_TABLE.
# Hint 3 shows the canonical loop.

row_check = run_sql(f"SELECT count(*), sum(length(page_text)) FROM {RAW_TEXT_TABLE}")
if row_check["rows"]:
    rows_count = int(row_check["rows"][0][0])
    chars_total = int(row_check["rows"][0][1] or 0)
    print(f"raw_text: {rows_count} pages, {chars_total} total characters")
else:
    print("raw_text: empty")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: source PDF parsed into the raw_text staging table with at
# least one page and more than 1000 characters of extracted text. Confirms
# the volume upload worked AND the PdfReader actually pulled text content
# (an empty-content PDF would clear row count but fail the character bar).


def checkpoint_1(session):
    try:
        sql = (
            "SELECT count(*), coalesce(sum(length(page_text)), 0) "
            f"FROM {RAW_TEXT_TABLE}"
        )
        result = run_sql(sql)
        if not result["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{RAW_TEXT_TABLE} returned no rows from count(*). Did the "
                    f"CREATE TABLE plus INSERT in Task 1 run?"
                ),
            )
        rows_count = int(result["rows"][0][0])
        chars_total = int(result["rows"][0][1] or 0)
        if rows_count == 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{RAW_TEXT_TABLE} has 0 rows. The pypdf loop must INSERT one "
                    f"row per PDF page (page_index INT, page_text STRING)."
                ),
            )
        if chars_total < 1000:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{RAW_TEXT_TABLE} has {rows_count} rows but only {chars_total} "
                    f"total characters of text; expected more than 1000 for a 20-page "
                    f"PDF. Check whether page.extract_text() returned None for most "
                    f"pages (scanned-image PDFs return None and need OCR; the "
                    f"fixture PDF is born-digital so this should not happen)."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint reads `count(*)` and `sum(length(page_text))` from `raw_text`. Three failure modes: the table is missing (CREATE TABLE never ran), the table is empty (the INSERT loop never executed), or the table has rows but the page_text column is empty / very short (PdfReader returned None for most pages or you forgot to write the text into the column). Read the error message; it names the case.

</details>

<details><summary>Hint 2 (direction)</summary>

Three SQL statements plus one Python loop:

- `CREATE SCHEMA IF NOT EXISTS workspace.default.labrun_chunking` then `ALTER SCHEMA ... SET TAGS ('labrun_lab_slug' = 'document-loading-and-chunking-strategies')`.
- `CREATE VOLUME IF NOT EXISTS workspace.default.labrun_chunking.source_docs`.
- `CREATE TABLE IF NOT EXISTS workspace.default.labrun_chunking.raw_text (page_index INT, page_text STRING) USING DELTA`.

Then `reader = PdfReader(FIXTURE_PDF_PATH)`, walk `reader.pages` with `enumerate`, call `page.extract_text()`, escape single quotes in the text, batch the rows into one `INSERT INTO ... VALUES (1, '...'), (2, '...'), ...` statement. A single multi-row INSERT is faster than 20 separate INSERTs and is well under the 16 MB statement-size ceiling.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
run_sql(f"CREATE SCHEMA IF NOT EXISTS {LAB_SCHEMA_FQN}")
run_sql(
    f"ALTER SCHEMA {LAB_SCHEMA_FQN} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)
run_sql(f"CREATE VOLUME IF NOT EXISTS {LAB_VOLUME_FQN}")
run_sql(
    f"CREATE TABLE IF NOT EXISTS {RAW_TEXT_TABLE} ("
    f"  page_index INT, page_text STRING"
    f") USING DELTA"
)

reader = PdfReader(FIXTURE_PDF_PATH)
rows = []
for idx, page in enumerate(reader.pages):
    text = page.extract_text() or ""
    safe = text.replace("'", "''")
    rows.append(f"({idx}, '{safe}')")
if rows:
    run_sql(f"INSERT INTO {RAW_TEXT_TABLE} VALUES " + ", ".join(rows))
```

</details>

**Wallet check.** Zero embedding calls so far. The PDF parse and the four metastore writes are free. Token spend: $0.00. Your morning coffee still wins by infinity.

## Task 2: Run the PDF through three chunking strategies, embed every chunk, persist three tables

The three strategies, plain English:

- **Fixed-size 512 with 50 overlap.** Concatenate every page into one long string. Slide a 512-character window across it, stepping by 462 characters (so adjacent chunks overlap by 50). Cheap, deterministic, ignores document structure entirely.
- **Recursive character splitter, markdown-aware separators.** `RecursiveCharacterTextSplitter(separators=["\n\n", "\n", ". ", " "], chunk_size=512, chunk_overlap=50)`. It splits on the highest-ranked separator that produces chunks under the size cap, falling back through paragraph break, line break, sentence break, word break. Preserves more structure than fixed-size, costs about the same.
- **Semantic chunking.** Split the text into sentences. Embed each sentence individually. Walk neighbor pairs; where the cosine similarity between sentence N and sentence N+1 drops below a threshold (0.5), close the current chunk and start a new one. Boundaries land on topic shifts, not character counts. Costs more (one embedding call per sentence) but the chunks track meaning.

Build three functions: `chunk_fixed(text, size=512, overlap=50) -> list[str]`, `chunk_recursive(text) -> list[str]`, `chunk_semantic(sentences, threshold=0.5) -> list[str]`. For each strategy:

1. Produce the chunk list.
2. Call `embed_texts(chunks)` to get a `(N, 1024)` numpy array.
3. Persist to the matching Delta table with schema `(chunk_id STRING, chunk_index INT, chunk_text STRING, embedding ARRAY<FLOAT>)`. Use a UUID-style `chunk_id` so cross-strategy comparisons cannot collide.

Checkpoint 2 reads each of the three tables, confirms at least 10 rows per table, and asserts `size(embedding) = 1024` on every row. A wrong-dimension embedding (for example, using `text-embedding-3-small` from OpenAI which is 1536-dim, or returning the raw text instead of the vector) fails immediately.

In [ ]:
# Task 2: Three chunking functions, three embedding loops, three Delta tables.
import uuid

# Pull the concatenated text from raw_text. ORDER BY page_index keeps the
# document flow correct so sliding-window and recursive splitters see
# paragraphs in the right sequence.
rows_result = run_sql(
    f"SELECT page_index, page_text FROM {RAW_TEXT_TABLE} ORDER BY page_index"
)
full_text = "\n\n".join((row[1] or "") for row in rows_result["rows"])
print(f"Loaded {len(full_text)} characters of source text.")


def chunk_fixed(text, size=512, overlap=50):
    # YOUR CODE: Slide a window of `size` characters across `text`, stepping
    # by (size - overlap). Return a list[str] of the resulting chunks.
    # Stop when the start index passes len(text).
    raise NotImplementedError("Replace with the sliding-window implementation.")


def chunk_recursive(text):
    # YOUR CODE: Build a RecursiveCharacterTextSplitter with separators=
    # ['\n\n', '\n', '. ', ' '], chunk_size=512, chunk_overlap=50; return
    # splitter.split_text(text).
    raise NotImplementedError("Replace with the langchain splitter call.")


def chunk_semantic(sentences, threshold=0.5):
    # YOUR CODE: Embed every sentence with embed_texts(sentences). Walk
    # neighbor pairs; if cosine(emb[i], emb[i+1]) < threshold, close the
    # current chunk after sentence i and start a new one with sentence i+1.
    # Return a list[str] where each entry is the joined sentences in a chunk.
    raise NotImplementedError("Replace with the semantic-boundary implementation.")


def _persist_chunks(table_fqn, chunks, embeddings):
    """Drop and recreate the table, then INSERT rows. Idempotent across re-runs."""
    run_sql(f"DROP TABLE IF EXISTS {table_fqn}")
    run_sql(
        f"CREATE TABLE {table_fqn} ("
        f"  chunk_id STRING, chunk_index INT, chunk_text STRING, embedding ARRAY<FLOAT>"
        f") USING DELTA"
    )
    if not chunks:
        return
    values = []
    for idx, (text_, vec) in enumerate(zip(chunks, embeddings)):
        safe = (text_ or "").replace("'", "''")
        cid = str(uuid.uuid4())
        vec_literal = "array(" + ", ".join(f"CAST({float(v):.6f} AS FLOAT)" for v in vec) + ")"
        values.append(f"('{cid}', {idx}, '{safe}', {vec_literal})")
    # Batch INSERTs by 25 to keep statement size below the 16 MB ceiling.
    BATCH = 25
    for start in range(0, len(values), BATCH):
        chunk_values = values[start:start + BATCH]
        run_sql(f"INSERT INTO {table_fqn} VALUES " + ", ".join(chunk_values))


# Strategy 1: fixed-size.
fixed_chunks = chunk_fixed(full_text, size=512, overlap=50)
fixed_embeds = embed_texts(fixed_chunks)
_persist_chunks(CHUNKS_FIXED_TABLE, fixed_chunks, fixed_embeds)
print(f"chunks_fixed: {len(fixed_chunks)} chunks, embedding shape {fixed_embeds.shape}")

# Strategy 2: recursive character splitter.
recursive_chunks = chunk_recursive(full_text)
recursive_embeds = embed_texts(recursive_chunks)
_persist_chunks(CHUNKS_RECURSIVE_TABLE, recursive_chunks, recursive_embeds)
print(f"chunks_recursive: {len(recursive_chunks)} chunks, embedding shape {recursive_embeds.shape}")

# Strategy 3: semantic chunking.
sentence_pattern = re.compile(r"(?<=[.!?])\s+")
sentences = [s.strip() for s in sentence_pattern.split(full_text) if s.strip()]
# Cap sentence count at 200 to keep the semantic-pass embedding spend bounded
# (a 20-page PDF should land well under this).
sentences = sentences[:200]
semantic_chunks = chunk_semantic(sentences, threshold=0.5)
semantic_embeds = embed_texts(semantic_chunks)
_persist_chunks(CHUNKS_SEMANTIC_TABLE, semantic_chunks, semantic_embeds)
print(f"chunks_semantic: {len(semantic_chunks)} chunks, embedding shape {semantic_embeds.shape}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: each of the three chunks tables has at least 10 rows and
# every row has size(embedding) = 1024 (the databricks-gte-large-en
# dimension). A wrong-dimension result usually means the wrong embedding
# model was called or the array was stored as text instead of ARRAY<FLOAT>.


def checkpoint_2(session):
    try:
        tables = [
            (CHUNKS_FIXED_TABLE, "chunks_fixed"),
            (CHUNKS_RECURSIVE_TABLE, "chunks_recursive"),
            (CHUNKS_SEMANTIC_TABLE, "chunks_semantic"),
        ]
        for fqn, label in tables:
            count_sql = f"SELECT count(*) FROM {fqn}"
            count_res = run_sql(count_sql)
            if not count_res["rows"]:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"{fqn} ({label}) returned no rows on count(*).",
                )
            row_count = int(count_res["rows"][0][0])
            if row_count < 10:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{fqn} ({label}) has {row_count} rows; expected at least 10. "
                        f"A 20-page PDF should produce well over 10 chunks at the "
                        f"configured chunk sizes."
                    ),
                )

            dim_sql = (
                f"SELECT min(size(embedding)), max(size(embedding)), "
                f"sum(CASE WHEN embedding IS NULL THEN 1 ELSE 0 END) FROM {fqn}"
            )
            dim_res = run_sql(dim_sql)
            if not dim_res["rows"]:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"{fqn} ({label}) returned no rows on size(embedding).",
                )
            min_sz = dim_res["rows"][0][0]
            max_sz = dim_res["rows"][0][1]
            null_count = int(dim_res["rows"][0][2] or 0)
            if null_count > 0:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{fqn} ({label}) has {null_count} rows with NULL embedding. "
                        f"Every row must carry a 1024-dim vector from "
                        f"databricks-gte-large-en."
                    ),
                )
            if min_sz != EMBEDDING_DIM or max_sz != EMBEDDING_DIM:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{fqn} ({label}) embedding sizes range [{min_sz}, {max_sz}]; "
                        f"expected exactly {EMBEDDING_DIM} on every row. Confirm the "
                        f"embedding model is {EMBEDDING_MODEL} and the embedding column "
                        f"type is ARRAY<FLOAT>, not STRING."
                    ),
                )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Three failure modes: (a) one of the three functions raises `NotImplementedError`, so the table is missing entirely; (b) the function runs but produces fewer than 10 chunks (chunk_size too large, threshold too tight, or the input text too short); (c) the function and the persist both run but `size(embedding)` is wrong (most often because the embedding column was stored as a STRING JSON blob instead of ARRAY<FLOAT>, or because someone reached for an OpenAI 1536-dim model by mistake). The checkpoint error message names which table and which case.

</details>

<details><summary>Hint 2 (direction)</summary>

Fixed: `while start < len(text): yield text[start:start+size]; start += size - overlap`. Recursive: `RecursiveCharacterTextSplitter(separators=['\n\n', '\n', '. ', ' '], chunk_size=512, chunk_overlap=50).split_text(text)`. Semantic: embed every sentence, compare neighbor pairs by `cos_sim(emb[i], emb[i+1])`, split where similarity dips below 0.5, join sentences within each chunk with spaces. The `_persist_chunks` helper does the INSERT for you; do not change its signature. Use the existing `embed_texts()` and `cosine_top_k()` helpers; do not call the OpenAI client directly here.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
def chunk_fixed(text, size=512, overlap=50):
    if not text:
        return []
    chunks = []
    step = max(1, size - overlap)
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += step
    return chunks


def chunk_recursive(text):
    splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", " "],
        chunk_size=512,
        chunk_overlap=50,
    )
    return splitter.split_text(text)


def chunk_semantic(sentences, threshold=0.5):
    if not sentences:
        return []
    sent_embeds = embed_texts(sentences)
    norms = np.linalg.norm(sent_embeds, axis=1)
    norms = np.where(norms == 0, 1.0, norms)
    chunks = []
    current = [sentences[0]]
    for i in range(len(sentences) - 1):
        sim = float(np.dot(sent_embeds[i], sent_embeds[i + 1]) / (norms[i] * norms[i + 1]))
        if sim < threshold:
            chunks.append(" ".join(current))
            current = [sentences[i + 1]]
        else:
            current.append(sentences[i + 1])
    if current:
        chunks.append(" ".join(current))
    return chunks
```

</details>

**Wallet check.** The semantic strategy embedded every sentence on the way in (cap of 200) plus every chunk on the way out; the other two only embedded chunks. Total embedding calls so far: roughly 150 batched into about 5 to 7 API calls (batch size 32). At a few hundred tokens per call against `databricks-gte-large-en`, total spend is a fraction of a cent. Token spend: $0.00 to $0.01. The thing this lab buys with your patience is the recursive splitter's separator debugging.

## Task 3: Compute recall@3 for each strategy on the same five-question evaluation set

The evaluation set is hand-built against this specific fixture PDF and ships with the lab as the `EVAL_SET` list (five questions, each paired with a unique substring that appears in the PDF). A retrieved chunk counts as a hit for a question if the ground-truth keyword appears anywhere in the chunk text. Recall@3 for a strategy is the fraction of the five questions where the keyword appears in at least one of the top-3 chunks retrieved by that strategy.

Steps:

1. Persist the five eval rows to `workspace.default.labrun_chunking.eval_set` with schema `(question_id INT, question STRING, ground_truth_span STRING)`.
2. For each strategy: load the chunk_text + embedding columns from that strategy's table into memory; for each of the five questions, embed the question (one call per question), find the top-3 chunks by cosine similarity, check whether the ground-truth keyword (case-insensitive) appears in any of the top-3 chunk_text values, accumulate hits.
3. Recall@3 = hits / 5 for that strategy.
4. Persist three rows to `workspace.default.labrun_chunking.chunking_results` with schema `(strategy STRING, recall_at_3 FLOAT, is_winner BOOLEAN, justification_text STRING)`. Leave `is_winner = FALSE` and `justification_text = ''` for now; Task 4 fills them in.

Checkpoint 3 reads the three rows, asserts every `recall_at_3` is between 0.0 and 1.0, then independently re-embeds one question and re-scores against one strategy's table inside the validator itself. If your scores are faked or computed without normalization, the validator's re-computation will disagree by more than 0.001 and the checkpoint fails.

In [ ]:
# Task 3: persist the eval set, score recall@3 per strategy, persist results.

# Persist the EVAL_SET fixture into a Delta table.
run_sql(f"DROP TABLE IF EXISTS {EVAL_SET_TABLE}")
run_sql(
    f"CREATE TABLE {EVAL_SET_TABLE} ("
    f"  question_id INT, question STRING, ground_truth_span STRING"
    f") USING DELTA"
)
eval_values = []
for qid, q_text, gt in EVAL_SET:
    q_safe = q_text.replace("'", "''")
    gt_safe = gt.replace("'", "''")
    eval_values.append(f"({qid}, '{q_safe}', '{gt_safe}')")
run_sql(f"INSERT INTO {EVAL_SET_TABLE} VALUES " + ", ".join(eval_values))
print(f"eval_set: {len(EVAL_SET)} rows")


def _load_chunks_for_scoring(table_fqn):
    """Pull (chunk_text, embedding) into memory as a list and an ndarray."""
    res = run_sql(
        f"SELECT chunk_text, embedding FROM {table_fqn} ORDER BY chunk_index"
    )
    texts = [row[0] or "" for row in res["rows"]]
    vectors = np.asarray([row[1] for row in res["rows"]], dtype=np.float32)
    return texts, vectors


def _score_recall_at_3(table_fqn):
    """Return float recall@3 across EVAL_SET for the given strategy table."""
    # YOUR CODE: Load chunks via _load_chunks_for_scoring(table_fqn). For each
    # (qid, question, ground_truth) in EVAL_SET, embed the question with
    # embed_texts([question])[0], call cosine_top_k(q_vec, chunk_matrix, k=3)
    # to get top-3 indices, check whether ground_truth.lower() is a substring
    # of any of those three chunk_text values (case-insensitive). Count hits,
    # divide by len(EVAL_SET), round to 3 decimals.
    raise NotImplementedError("Replace with the recall@3 scoring loop.")


strategy_tables = [
    ("fixed", CHUNKS_FIXED_TABLE),
    ("recursive", CHUNKS_RECURSIVE_TABLE),
    ("semantic", CHUNKS_SEMANTIC_TABLE),
]
strategy_scores = {}
for strategy_name, fqn in strategy_tables:
    score = _score_recall_at_3(fqn)
    strategy_scores[strategy_name] = float(score)
    print(f"recall@3 for {strategy_name}: {score:.3f}")

# Persist three results rows. is_winner and justification_text get filled in
# in Task 4. Initialize them as FALSE and empty so Task 4's UPDATE has a
# known starting point.
run_sql(f"DROP TABLE IF EXISTS {RESULTS_TABLE}")
run_sql(
    f"CREATE TABLE {RESULTS_TABLE} ("
    f"  strategy STRING, recall_at_3 FLOAT, is_winner BOOLEAN, justification_text STRING"
    f") USING DELTA"
)
values = []
for strategy_name, _ in strategy_tables:
    score = strategy_scores[strategy_name]
    values.append(f"('{strategy_name}', CAST({score:.6f} AS FLOAT), FALSE, '')")
run_sql(f"INSERT INTO {RESULTS_TABLE} VALUES " + ", ".join(values))
print(f"chunking_results: 3 rows written")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: chunking_results has exactly 3 rows, each with a recall_at_3
# between 0.0 and 1.0; eval_set has exactly 5 rows; AND an independent
# re-computation of recall@3 for one strategy (chunks_fixed) inside the
# validator matches the persisted score within 0.001 tolerance. The
# re-computation is what makes the checkpoint reject faked scores.


def checkpoint_3(session):
    try:
        eval_count_res = run_sql(f"SELECT count(*) FROM {EVAL_SET_TABLE}")
        eval_count = int(eval_count_res["rows"][0][0]) if eval_count_res["rows"] else 0
        if eval_count != 5:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{EVAL_SET_TABLE} has {eval_count} rows; expected exactly 5. "
                    f"The five-question fixture must be persisted verbatim."
                ),
            )

        results_res = run_sql(
            f"SELECT strategy, recall_at_3 FROM {RESULTS_TABLE} ORDER BY strategy"
        )
        if not results_res["rows"] or len(results_res["rows"]) != 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{RESULTS_TABLE} has {len(results_res['rows'])} rows; expected "
                    f"exactly 3 (one per strategy: fixed, recursive, semantic)."
                ),
            )
        seen_strategies = set()
        for row in results_res["rows"]:
            strategy = row[0]
            recall_value = row[1]
            if recall_value is None:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"{RESULTS_TABLE}.recall_at_3 is NULL for strategy {strategy!r}.",
                )
            recall_value = float(recall_value)
            if recall_value < 0.0 or recall_value > 1.0:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{RESULTS_TABLE}.recall_at_3 for {strategy!r} is {recall_value}; "
                        f"must be between 0.0 and 1.0."
                    ),
                )
            seen_strategies.add(strategy)
        for required in ("fixed", "recursive", "semantic"):
            if required not in seen_strategies:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{RESULTS_TABLE} is missing a row for strategy {required!r}. "
                        f"Expected strategy values: 'fixed', 'recursive', 'semantic'."
                    ),
                )

        # Independent re-computation for 'fixed'. Reload its chunks, embed
        # every question fresh, score against the validator's own logic, and
        # compare against the persisted value within tolerance 0.001.
        chunk_res = run_sql(
            f"SELECT chunk_text, embedding FROM {CHUNKS_FIXED_TABLE} ORDER BY chunk_index"
        )
        chunk_texts = [row[0] or "" for row in chunk_res["rows"]]
        chunk_matrix = np.asarray([row[1] for row in chunk_res["rows"]], dtype=np.float32)
        if chunk_matrix.shape[0] == 0:
            return CheckpointResult(
                status="fail",
                error_reason=f"{CHUNKS_FIXED_TABLE} has no rows for independent scoring.",
            )

        hits = 0
        for qid, question, ground_truth in EVAL_SET:
            q_vec = embed_texts([question])[0]
            top_idx = cosine_top_k(q_vec, chunk_matrix, k=3)
            joined = " ".join(chunk_texts[i] for i in top_idx).lower()
            if ground_truth.lower() in joined:
                hits += 1
        independent_recall = round(hits / len(EVAL_SET), 3)

        persisted_fixed_res = run_sql(
            f"SELECT recall_at_3 FROM {RESULTS_TABLE} WHERE strategy = 'fixed'"
        )
        persisted = float(persisted_fixed_res["rows"][0][0])
        if abs(persisted - independent_recall) > 0.001:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Independent recall@3 for 'fixed' computed by the validator is "
                    f"{independent_recall}, but the persisted value is {persisted}. "
                    f"Difference exceeds 0.001 tolerance. Confirm cosine similarity "
                    f"uses normalized vectors and the substring match is "
                    f"case-insensitive against the ground-truth keyword."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Two failure modes worth knowing about. First, the validator counts the eval_set rows and the chunking_results rows; missing rows or extra rows fail before the validator does any scoring. Second, the validator re-runs recall@3 for the fixed strategy itself and compares against your persisted value within 0.001. If those two scores disagree, your scoring loop is either using unnormalized cosine, doing a case-sensitive keyword match, or returning a fake number.

</details>

<details><summary>Hint 2 (direction)</summary>

For each strategy table: `texts, vectors = _load_chunks_for_scoring(fqn)`, then loop over EVAL_SET. For each `(qid, question, ground_truth)`: embed the question with `embed_texts([question])[0]`, get top-3 indices with `cosine_top_k(q_vec, vectors, k=3)`, lowercase the joined top-3 chunk_text values and check whether `ground_truth.lower()` appears as a substring. Count hits, divide by 5, round to 3 decimals. The helper `cosine_top_k` is already normalized; do not roll your own dot product here or your numbers will drift from the validator.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
def _score_recall_at_3(table_fqn):
    chunk_texts, chunk_matrix = _load_chunks_for_scoring(table_fqn)
    if chunk_matrix.shape[0] == 0:
        return 0.0
    hits = 0
    for qid, question, ground_truth in EVAL_SET:
        q_vec = embed_texts([question])[0]
        top_idx = cosine_top_k(q_vec, chunk_matrix, k=3)
        joined = " ".join(chunk_texts[i] for i in top_idx).lower()
        if ground_truth.lower() in joined:
            hits += 1
    return round(hits / len(EVAL_SET), 3)
```

</details>

**Wallet check.** Task 3 added 15 question embeddings (5 questions x 3 strategies) plus the validator re-embeds 5 more questions for its independent check. About 20 more embedding calls, all small. Cumulative token spend is still well under 3 cents. Your morning coffee continues to be the more expensive item by a factor of about 100.

## Task 4: Pick the winner, write a one-paragraph justification, persist the decision

The winner is the strategy with the highest `recall_at_3`. If two strategies tie, prefer the recursive splitter; it is the cheaper of the two structural strategies and ties happen because the eval set is small.

Write a justification paragraph that does two things:

1. Names the winning strategy.
2. Names at least one structural property of the source PDF that explains why this strategy beat the other two on this document. Useful structural-property words: `paragraph`, `page break`, `markdown`, `section`, `header`, `sentence boundary`, `semantic`. The validator checks that at least one of these appears in your text (case-insensitive).

Persist via two UPDATE statements: set `is_winner = TRUE` on the winning row, set `justification_text` on the winning row to your paragraph (over 80 characters, contains at least one structural keyword).

In [ ]:
# Task 4: pick the winner, write the justification, UPDATE the results row.

scores_res = run_sql(
    f"SELECT strategy, recall_at_3 FROM {RESULTS_TABLE} ORDER BY recall_at_3 DESC, strategy ASC"
)
ranked = [(row[0], float(row[1])) for row in scores_res["rows"]]
print("Ranked recall@3 across strategies:")
for strategy, score in ranked:
    print(f"  {strategy}: {score:.3f}")

# YOUR CODE: Pick the winning strategy (highest recall_at_3 from `ranked`).
# Apply the tie-break rule from the task markdown: if two strategies share
# the top score, prefer 'recursive' over 'fixed' over 'semantic'. Assign the
# winner to `winner_strategy`.
winner_strategy = None

# YOUR CODE: Write a one-paragraph justification (over 80 characters) that
# names the structural property of this PDF that explains the win. Include
# at least one of these structural-property words: paragraph, page break,
# markdown, section, header, sentence boundary, semantic. Assign it to
# `justification`.
justification = None

if not winner_strategy or not justification:
    print("Set winner_strategy and justification, then re-run this cell.")
    raise SystemExit(1)

winner_safe = winner_strategy.replace("'", "''")
justification_safe = justification.replace("'", "''")

# Reset every row to is_winner=FALSE first so a Task 4 re-run after a wrong
# initial choice cannot leave two winners flagged.
run_sql(f"UPDATE {RESULTS_TABLE} SET is_winner = FALSE")
run_sql(
    f"UPDATE {RESULTS_TABLE} "
    f"SET is_winner = TRUE, justification_text = '{justification_safe}' "
    f"WHERE strategy = '{winner_safe}'"
)
print(f"Marked winner: {winner_strategy}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: chunking_results has exactly one row with is_winner = TRUE;
# the winning row also has a justification_text over 80 characters that
# contains at least one structural-property keyword.

STRUCTURAL_KEYWORDS = (
    "paragraph",
    "page break",
    "markdown",
    "section",
    "header",
    "sentence boundary",
    "semantic",
)


def checkpoint_4(session):
    try:
        winners_res = run_sql(
            f"SELECT strategy, recall_at_3, is_winner, justification_text "
            f"FROM {RESULTS_TABLE} WHERE is_winner = TRUE"
        )
        winner_rows = winners_res["rows"]
        if len(winner_rows) == 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No row in {RESULTS_TABLE} has is_winner = TRUE. Set is_winner "
                    f"= TRUE on the strategy with the highest recall_at_3."
                ),
            )
        if len(winner_rows) > 1:
            winners = ", ".join(str(r[0]) for r in winner_rows)
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{RESULTS_TABLE} has {len(winner_rows)} rows with is_winner = "
                    f"TRUE ({winners}). Exactly one row may be flagged as winner. "
                    f"Reset others to FALSE."
                ),
            )

        strategy, recall_value, _is_winner, justification_text = winner_rows[0]
        # Confirm the flagged winner actually has the top recall@3 score.
        all_res = run_sql(
            f"SELECT strategy, recall_at_3 FROM {RESULTS_TABLE} ORDER BY recall_at_3 DESC"
        )
        top_score = float(all_res["rows"][0][1])
        if float(recall_value) < top_score - 1e-6:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Row flagged is_winner = TRUE has recall_at_3 = {recall_value}, "
                    f"but the highest recall_at_3 in the table is {top_score}. The "
                    f"winning row must hold the top score."
                ),
            )

        justification_text = (justification_text or "").strip()
        if len(justification_text) <= 80:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"justification_text on the winning row is {len(justification_text)} "
                    f"characters; must be over 80. Write a paragraph that names the "
                    f"structural property of this PDF that explains the win."
                ),
            )
        lower = justification_text.lower()
        if not any(keyword in lower for keyword in STRUCTURAL_KEYWORDS):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"justification_text on the winning row does not name a structural "
                    f"property. Include at least one of: "
                    f"{', '.join(STRUCTURAL_KEYWORDS)}."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Three failure modes: (a) no row has is_winner = TRUE (you forgot the UPDATE), (b) two rows are flagged (the reset-to-FALSE step did not run), or (c) the justification is either too short (under 80 characters) or does not contain a structural keyword. The error message names which case.

</details>

<details><summary>Hint 2 (direction)</summary>

Walk `ranked` (already sorted by recall_at_3 descending). The top entry is the winner unless there is a tie at the top; the tie-break rule in the task markdown is recursive > fixed > semantic. For the justification, name the structural property of the source PDF: this document has paragraph breaks the recursive splitter respects, page-break headers the fixed splitter ignores, and topic shifts the semantic splitter overweights. Pick whichever sentence is true for your winner and include one of the structural-property keywords verbatim so the validator's substring check passes. The UPDATE statements are vanilla SQL; do not skip the reset-to-FALSE pass or a previous tie can leave two winners flagged.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4 (recursive wins on the fixture PDF; substitute your actual winner if your numbers come out different):

```python
tie_break_order = {"recursive": 0, "fixed": 1, "semantic": 2}
top_score = ranked[0][1]
top_candidates = [s for s, score in ranked if abs(score - top_score) < 1e-6]
winner_strategy = sorted(top_candidates, key=lambda s: tie_break_order.get(s, 99))[0]

justification = (
    "The recursive splitter wins on this PDF because the document is structured around "
    "paragraph and section boundaries; splitting on '\\n\\n' before falling back to "
    "sentence boundary separators keeps each chunk anchored to one idea, while the "
    "fixed-size splitter cuts across page breaks and dilutes retrievable context."
)
```

If `semantic` or `fixed` wins on your run, rewrite the paragraph to name the structural property that strategy exploits (semantic boundary shifts for the semantic strategy, sliding-window coverage for the fixed-size strategy) and keep at least one structural keyword in the text.

</details>

**Wallet check.** Task 4 added two metastore UPDATE statements and zero embedding calls. Cumulative session spend is still under 3 cents on tokens, with the Starter warehouse contributing a handful of warehouse-seconds for free. Your morning coffee retains its lead.

## Cleanup

Time to tear it all down. The cell below runs through your manifest in reverse-creation order (chunking_results first, then eval_set, then the three chunks tables, then raw_text, then the volume contents and the volume, then the schema with `CASCADE`), then double-checks UC information_schema to confirm everything is gone. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order using the inline Databricks adapter. No critical (hourly-billed)
# resources in this lab, so the canonical summary always reports zero
# critical destructions.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.00 to $0.03.** Foundation Model API embedding calls dominate the spend; everything else (Starter warehouse, UC storage, metastore writes) sits in Free Edition's swallow-all column. The seven Delta tables, the volume contents, the volume, and the schema were all destroyed by the cleanup cell above. Your daily compute quota took a small dip and resets at 00:00 UTC.

## Reflection

These are not graded. They are for you.

1. Walk through what the recursive splitter did differently from the fixed-size splitter on the page-break boundary. Why did splitting on `\n\n` first, then `\n`, then `. `, then ` ` preserve more retrievable chunks than a flat 512-character split?

2. Semantic chunking uses embedding cosine similarity to find boundaries. What is the failure mode when consecutive sentences are topically similar but functionally different (a procedural step followed by a warning about that step)? When would you prefer fixed-size chunking even though it underperforms on this PDF?

## Exam-Style Practice

**Question 1 (Domain 2, chunking strategy per the official sample-question pattern):**

A GenAI engineer is loading 150 million chunks into a Mosaic AI Vector Search standard endpoint that caps at 100 million embeddings. Which TWO actions reduce the chunk count?

A. Increase the document chunk size.

B. Decrease the overlap between chunks.

C. Decrease the document chunk size.

D. Increase the overlap between chunks.

E. Use a smaller embedding model.

<details><summary>Show answer</summary>

**Correct: A and B.**

- A is correct: a larger chunk size means fewer chunks per document; total chunk count drops.
- B is correct: less overlap means fewer overlapping chunks at chunk boundaries; total chunk count drops.
- C is wrong: smaller chunks means MORE chunks per document; count rises.
- D is wrong: more overlap means MORE chunks; count rises.
- E is wrong: embedding model size affects embedding dimension and cost, not the chunk count. The chunk count is determined by the chunker, not the embedder.

This is verbatim the Question 1 pattern from the official sample-question set in the March 2026 exam guide.

</details>

**Question 2 (Domain 2, advanced chunking and re-ranking):**

A GenAI engineer's RAG application returns the right chunks at top-10 but the LLM only ever reads top-3. Recall@10 is 0.9; recall@3 is 0.45. Which approach improves the top-3 recall without changing the chunking strategy?

A. Add a cross-encoder reranker after the vector retriever to reorder the top-10 chunks by query relevance.

B. Increase the LLM context window so it reads all 10 chunks.

C. Decrease the chunk size so more chunks fit in the top-3.

D. Increase the chunk overlap so adjacent chunks contain similar content.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: a cross-encoder reranker is the canonical fix for 'right chunks at top-K but wrong order in top-N'. Cross-encoders score each (query, chunk) pair jointly and produce a more accurate ranking than the original bi-encoder cosine similarity. This is the role of re-ranking the exam guide names in Section 2.
- B is wrong: changing the LLM context window does not change the retriever's quality and burns much more cost-per-query.
- C is wrong: the chunking strategy is fixed in this scenario; the question asks for a non-chunking fix.
- D is wrong: more chunk overlap is a chunking-strategy change and does not address the ranking-quality problem.

</details>